# Protótipo 1 — Árvores para prever a glicemia em 30 minutos

**Pergunta:** dado o que foi registrado até o instante *t*, qual será a glicemia em *t + 30 min*? E **registrar carboidratos e bolus ajuda** em relação a usar só a glicemia?

**Na prática:** cada linha da tabela de treino é um instante. Ela reúne as últimas 12 leituras de CGM (1 hora) e um resumo dos carboidratos e bolus recentes. Um `HistGradientBoostingRegressor` prevê a glicemia 30 min depois. Treinamos duas versões (só CGM × CGM + eventos) e comparamos com a **persistência** (prever `G(t)`).

Neste notebook, o que a especificação chama de `--quick` é a variável `QUICK` da primeira célula de código. Sugestão: rode com `QUICK = True`, confira, depois com `QUICK = False`.

**Fluxo:** (1) carregar e limpar → (2) grade de 5 min → (3) divisão temporal → (4) montar entradas/alvo → (5) treinar → (6) avaliar → (7) salvar.

## 0. Configuração

Aqui ficam o modo rápido, a semente, os participantes (Subject 1, 2 e 3) e o caminho dos dados. **Ajuste `DATA_DIR`** para a pasta `CGM Records` no seu Drive. Os resultados vão para `results/prototipo_01_arvores/`.

In [ ]:
import os, re, glob, json, time, platform, warnings, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib
warnings.filterwarnings("ignore")

# ----------------------------- CONFIGURAÇÃO -----------------------------
QUICK = True                 # True = execução pequena (rápida); False = execução completa
SEED = 42
SUBJECTS_WANTED = [1, 2, 3]  # None = todos os participantes encontrados
DATA_DIR = "/content/drive/MyDrive/AZT1D 2025/CGM Records"   # <-- ajuste para a sua pasta

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_OUT = "/content/drive/MyDrive"
except ImportError:          # rodando fora do Colab
    BASE_OUT = "."
OUT_DIR = os.path.join(BASE_OUT, "results", "prototipo_01_arvores")
os.makedirs(OUT_DIR, exist_ok=True)
print("QUICK =", QUICK, "| DATA_DIR =", DATA_DIR, "| OUT_DIR =", OUT_DIR)

## 1. Funções de preparação

Regras importantes (todas vêm da especificação e evitam vazamento de informação do futuro):

- **Grade de 5 min com intervalos `(t − 5 min, t]` rotulados no final.** Um evento às 10:04 pertence ao intervalo rotulado 10:05, portanto só "existe" a partir das 10:05. Isso equivale a `ceil('5min')` (arredondar para cima). Arredondar para baixo colocaria o evento às 10:00, *antes* de ele ser registrado — um vazamento.
- **CGM:** se o intervalo tem um único valor distinto válido, mantém; se tem valores conflitantes, vira ausente. **Nunca interpola** nem preenche lacunas.
- **Carboidratos e bolus:** soma dos registros distintos por intervalo (após remover duplicatas). **Zero = nenhum evento registrado**, não certeza de que nada foi ingerido/aplicado.
- Só `CarbSize` e `TotalBolusInsulinDelivered`. `FoodDelivered` não é carboidrato; `CorrectionDelivered` pode já estar dentro do bolus total; `Basal` fica de fora (semântica pendente).
- Se o arquivo não tiver coluna `CGM` (alias `Readings (CGM / BGM)`), o participante é excluído e o motivo é registrado.
- Glicemia inválida (≤ 0) vira ausente, **nunca zero**.

In [ ]:
def find_subject_files(data_dir):
    """Localiza os CSVs e identifica o participante pelo nome da pasta/arquivo (ex.: 'Subject 3')."""
    found = {}
    for f in glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True):
        m = re.search(r"Subject\s*_?(\d+)", f, flags=re.I)
        if m:
            found.setdefault(int(m.group(1)), f)
    return dict(sorted(found.items()))


def prepare_subject(path):
    """CSV bruto -> grade regular de 5 min com colunas cgm, carb, bolus. Devolve (grade, auditoria)."""
    audit = {}
    df = pd.read_csv(path)
    audit["linhas_brutas"] = len(df)

    # Alias ambíguo: sem coluna CGM não dá para garantir que os registros são CGM -> exclui
    if "CGM" not in df.columns:
        return None, {**audit, "excluido": "sem coluna 'CGM' (alias 'Readings (CGM / BGM)' não confirmado)"}

    n0 = len(df); df = df.drop_duplicates(); audit["duplicadas_removidas"] = n0 - len(df)
    df["EventDateTime"] = pd.to_datetime(df["EventDateTime"], errors="coerce")
    audit["datas_invalidas"] = int(df["EventDateTime"].isna().sum())
    df = df.dropna(subset=["EventDateTime"]).sort_values("EventDateTime")

    for c in ["CGM", "CarbSize", "TotalBolusInsulinDelivered"]:
        if c not in df.columns:
            df[c] = np.nan; audit[f"{c}_coluna_ausente"] = True
        raw = df[c]; num = pd.to_numeric(raw, errors="coerce")
        audit[f"{c}_nao_numerico"] = int((num.isna() & raw.notna()).sum())
        audit[f"{c}_ausente"] = int(raw.isna().sum())
        df[c] = num

    df.loc[df["CGM"] <= 0, "CGM"] = np.nan                       # glicemia inválida -> ausente (nunca zero)
    df.loc[df["CarbSize"] < 0, "CarbSize"] = np.nan
    df.loc[df["TotalBolusInsulinDelivered"] < 0, "TotalBolusInsulinDelivered"] = np.nan

    # Intervalo (t-5min, t] rotulado em t  ==  arredondar PARA CIMA para múltiplos de 5 min
    df["bin"] = df["EventDateTime"].dt.ceil("5min")
    g = df.groupby("bin")

    mm = g["CGM"].agg(["min", "max"])
    cgm = mm["min"].where(mm["min"] == mm["max"])                 # 1 único valor distinto -> mantém; conflito -> NaN
    audit["intervalos_cgm_conflitantes"] = int(((mm["min"] != mm["max"]) & mm["min"].notna()).sum())
    carb = g["CarbSize"].sum()                                    # soma dos registros distintos; NaN é ignorado
    bol = g["TotalBolusInsulinDelivered"].sum()

    idx = pd.date_range(df["bin"].min(), df["bin"].max(), freq="5min")
    grid = pd.DataFrame({"cgm": cgm.reindex(idx),
                         "carb": carb.reindex(idx).fillna(0.0),   # 0 = nenhum evento REGISTRADO
                         "bolus": bol.reindex(idx).fillna(0.0)}, index=idx)
    audit["passos_na_grade"] = len(grid)
    audit["passos_cgm_ausente"] = int(grid["cgm"].isna().sum())
    return grid, audit


def make_cuts(n):
    """60% treino / 20% validação / 20% teste, em ordem cronológica, como (início, fim) de índices da grade."""
    a, b = int(n * 0.6), int(n * 0.8)
    return {"treino": (0, a), "validacao": (a, b), "teste": (b, n)}


def valid_origins(cgm, part, back_cgm, back_ev, fwd):
    """Índices de origem t válidos dentro de UMA partição.
    - CGM sem nenhuma lacuna de t-back_cgm até t+fwd (segmento contínuo até o alvo);
    - todo o suporte (inclusive a janela de eventos back_ev e o alvo) dentro da partição."""
    s, e = part
    ok = ~np.isnan(cgm)
    cs = np.concatenate([[0], np.cumsum(ok)])
    i = np.arange(s + max(back_cgm, back_ev), e - fwd)
    lo, hi = i - back_cgm, i + fwd
    return i[(cs[hi + 1] - cs[lo]) == (hi - lo + 1)]


def prefix(a):
    """Soma acumulada com zero na frente: soma dos passos a..b = p[b+1] - p[a]."""
    return np.concatenate([[0.0], np.cumsum(a)])


def first_run(idx, min_len):
    """Primeiro trecho de origens consecutivas (passo 1 da grade) com pelo menos min_len pontos."""
    if len(idx) == 0: return None
    breaks = np.where(np.diff(idx) != 1)[0] + 1
    for run in np.split(np.arange(len(idx)), breaks):
        if len(run) >= min_len: return run
    return None


def md_table(df, nd=2):
    cols = list(df.columns)
    L = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        L.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |")
    return "\n".join(L)


def mae_rmse(y, p):
    if len(y) == 0: return np.nan, np.nan
    e = np.asarray(p) - np.asarray(y)
    return float(np.mean(np.abs(e))), float(np.sqrt(np.mean(e ** 2)))

## 2. Carregar os participantes e auditar

A tabela mostra, por participante, quantas linhas foram descartadas/convertidas — é a auditoria pedida na especificação.

In [ ]:
files = find_subject_files(DATA_DIR)
print("Participantes encontrados:", list(files))
if SUBJECTS_WANTED is not None:
    files = {k: v for k, v in files.items() if k in SUBJECTS_WANTED}
assert files, "Nenhum CSV encontrado: confira DATA_DIR (e se o Drive foi montado)."

t0 = time.time()
grids, audits = {}, {}
for sid, path in files.items():
    g, a = prepare_subject(path)
    audits[sid] = a
    if g is not None: grids[sid] = g
t_prep_load = time.time() - t0

pd.DataFrame(audits).T

Teste rápido da regra de alinhamento (um registro às 10:04 deve cair no intervalo 10:05):

In [ ]:
ts = pd.Series(pd.to_datetime(["2026-01-01 10:04:00", "2026-01-01 10:05:00", "2026-01-01 10:00:00", "2026-01-01 10:00:01"]))
print(ts.dt.ceil("5min").dt.strftime("%H:%M").tolist())   # esperado: ['10:05', '10:05', '10:00', '10:05']
assert ts.dt.ceil("5min").dt.strftime("%H:%M").tolist() == ["10:05", "10:05", "10:00", "10:05"]

## 3. Divisão temporal **antes** de gerar exemplos

Para cada participante, o período é cortado em ordem cronológica: primeiros 60% (treino), próximos 20% (validação) e últimos 20% (teste). Só depois geramos os exemplos, exigindo que **todo o suporte** (janela de 2 h anterior e o alvo) caia dentro da mesma partição. Assim nenhum exemplo atravessa a fronteira treino/teste. Os horários de corte ficam salvos.

Isso avalia o **futuro de pessoas já observadas**, não a generalização para pessoas novas.

In [ ]:
cuts, cut_times = {}, {}
for sid, g in grids.items():
    cuts[sid] = make_cuts(len(g))
    cut_times[sid] = {p: [str(g.index[s]), str(g.index[e - 1])] for p, (s, e) in cuts[sid].items()}
pd.DataFrame({sid: {p: f"{v[0]} -> {v[1]}" for p, v in d.items()} for sid, d in cut_times.items()}).T

## 4. Entradas e alvo

Em cada origem *t* (índice `i` da grade), com 5 min por passo:

| Grupo | Variáveis | Como sai da grade |
|---|---|---|
| Glicemia | 12 valores de `G(t−55)` a `G(t)` | `cgm[i-11 … i]` |
| Resumos | média, mínimo, máximo, `G(t) − G(t−15)` | sobre os 12 valores; `G[i] − G[i−3]` |
| Carboidratos | somas em 30, 60 e 120 min | 6, 12 e 24 passos terminando em `i` |
| Bolus | mesmas três somas | idem |
| **Alvo** | `G(t+30)` | `cgm[i+6]` (30 min = **6 passos**) |

As somas usam somas acumuladas (`prefix`) para serem rápidas. Os resumos de eventos são só **resumos de registros**, não insulina/carboidrato ativos.

In [ ]:
BACK_CGM, BACK_EV, FWD = 11, 23, 6        # 12 leituras; janela de eventos de 24 passos (2 h); alvo a 6 passos
G_COLS = [f"g_m{5*k}" for k in range(11, 0, -1)] + ["g_0"]
SUM_COLS = ["g_mean", "g_min", "g_max", "d15"]
EV_COLS = ["carb30", "carb60", "carb120", "bol30", "bol60", "bol120"]
FEAT_CGM = G_COLS + SUM_COLS
FEAT_EV = FEAT_CGM + EV_COLS


def build_p1(sid, grid, cuts_sid):
    cgm = grid["cgm"].values
    cc, bc = prefix(grid["carb"].values), prefix(grid["bolus"].values)
    parts = []
    for part, rng in cuts_sid.items():
        i = valid_origins(cgm, rng, BACK_CGM, BACK_EV, FWD)
        if len(i) == 0: continue
        G = np.stack([cgm[i - k] for k in range(11, -1, -1)], axis=1)       # (n, 12): G(t-55) ... G(t)
        d = pd.DataFrame(G, columns=G_COLS)
        d["g_mean"], d["g_min"], d["g_max"] = G.mean(1), G.min(1), G.max(1)
        d["d15"] = G[:, -1] - G[:, -4]                                       # G(t) - G(t-15 min)
        for n, w in [(30, 6), (60, 12), (120, 24)]:                          # soma dos passos i-w+1 ... i
            d[f"carb{n}"] = cc[i + 1] - cc[i + 1 - w]
            d[f"bol{n}"] = bc[i + 1] - bc[i + 1 - w]
        d["y"] = cgm[i + FWD]
        d["subject"], d["origin_idx"], d["particao"] = sid, i, part
        d["t"], d["t_target"] = grid.index[i], grid.index[i + FWD]
        parts.append(d)
    return pd.concat(parts, ignore_index=True) if parts else None


t0 = time.time()
all_ex, discarded = {}, {}
for sid, g in grids.items():
    ex = build_p1(sid, g, cuts[sid])
    if ex is None or set(ex.particao) != {"treino", "validacao", "teste"}:
        discarded[sid] = "sem exemplos válidos nas três partições"
    else:
        all_ex[sid] = ex
t_prep_ex = time.time() - t0

eligible = sorted(all_ex)
if QUICK: eligible = eligible[:3]                       # três primeiros elegíveis em ordem numérica
print("Elegíveis usados:", eligible, "| descartados:", discarded)
ex = pd.concat([all_ex[s] for s in eligible], ignore_index=True)
pd.crosstab(ex["subject"], ex["particao"])

### Limitação de treino no modo rápido

No modo rápido o treino é limitado a 20.000 exemplos (amostra com semente 42). **As duas variantes usam exatamente os mesmos exemplos**, só mudam as colunas.

In [ ]:
train = ex[ex.particao == "treino"].reset_index(drop=True)
if QUICK and len(train) > 20000:
    rng = np.random.default_rng(SEED)
    train = train.iloc[np.sort(rng.choice(len(train), 20000, replace=False))].reset_index(drop=True)
print("Exemplos de treino usados:", len(train))

## 5. Modelos

Três previsores, parâmetros fixos (sem busca de hiperparâmetros; árvores não precisam de padronização):

1. **Persistência:** `G(t+30) ≈ G(t)`.
2. **Árvores CGM:** só histórico e resumos de glicemia (16 variáveis).
3. **Árvores CGM + eventos:** as 16 anteriores + 6 resumos de carboidratos/bolus (22 variáveis).

O teste **não** é usado para ajustar nada.

In [ ]:
def new_model():
    return HistGradientBoostingRegressor(max_iter=100, max_leaf_nodes=15, learning_rate=0.05,
                                         early_stopping=False, random_state=42)

t0 = time.time()
m_cgm = new_model().fit(train[FEAT_CGM], train["y"])
m_ev = new_model().fit(train[FEAT_EV], train["y"])
t_train = time.time() - t0

t0 = time.time()
ex["pred_persistencia"] = ex["g_0"]
ex["pred_arvores_cgm"] = m_cgm.predict(ex[FEAT_CGM])
ex["pred_arvores_cgm_eventos"] = m_ev.predict(ex[FEAT_EV])
t_infer = time.time() - t0
print(f"preparação: {t_prep_load + t_prep_ex:.1f}s | treino: {t_train:.1f}s | inferência: {t_infer:.1f}s")

## 6. Verificações essenciais

- o alvo está exatamente 30 min depois da origem na grade;
- nenhum exemplo atravessa participante ou partição (todo o suporte de 2 h + alvo cabe dentro do intervalo da partição);
- as três previsões foram calculadas sobre as mesmas linhas.

(Entradas sem futuro: pela construção, só usamos índices `≤ i`; o alvo é o único valor em `i+6`.)

In [ ]:
assert ((ex["t_target"] - ex["t"]) == pd.Timedelta("30min")).all()
for sid in eligible:
    for part, (s, e) in cuts[sid].items():
        d = ex[(ex.subject == sid) & (ex.particao == part)]
        assert d["origin_idx"].min() - BACK_EV >= s and d["origin_idx"].max() + FWD <= e - 1, (sid, part)
assert ex[["pred_persistencia", "pred_arvores_cgm", "pred_arvores_cgm_eventos"]].notna().all().all()
# a variante com eventos usa exatamente as mesmas linhas: ambas predizem sobre o mesmo DataFrame `ex`
print("Verificações OK")

## 7. Métricas (MAE e RMSE em mg/dL)

Calculadas na validação e no teste, globalmente e por participante, em dois subconjuntos: **todos** os instantes e **carb_60min** (instantes com carboidrato registrado em `(t−60 min, t]`), para olhar o comportamento perto de refeições sem criar outro modelo.

In [ ]:
MODELS = {"persistencia": "pred_persistencia", "arvores_cgm": "pred_arvores_cgm",
          "arvores_cgm_eventos": "pred_arvores_cgm_eventos"}
rows = []
for part in ["validacao", "teste"]:
    d = ex[ex.particao == part]
    for subset, mask in [("todos", np.ones(len(d), bool)), ("carb_60min", (d["carb60"] > 0).values)]:
        ds = d[mask]
        groups = [("AGREGADO", ds)] + [(f"Subject {s}", ds[ds.subject == s]) for s in eligible]
        for who, sub in groups:
            for mname, col in MODELS.items():
                mae, rmse = mae_rmse(sub["y"], sub[col])
                rows.append(dict(particao=part, modelo=mname, participante=who, subconjunto=subset,
                                 n_exemplos=len(sub), MAE=mae, RMSE=rmse))
metricas = pd.DataFrame(rows)
metricas.to_csv(os.path.join(OUT_DIR, "metricas.csv"), index=False)

main = metricas[(metricas.participante == "AGREGADO")]
main.pivot_table(index=["particao", "subconjunto", "n_exemplos"], columns="modelo", values=["MAE", "RMSE"]).round(2)

In [ ]:
# Por participante (teste, todos os instantes)
metricas[(metricas.particao == "teste") & (metricas.subconjunto == "todos") & (metricas.participante != "AGREGADO")] \
    .pivot_table(index="participante", columns="modelo", values=["MAE", "RMSE"]).round(2)

In [ ]:
n_carb = metricas[(metricas.particao == "teste") & (metricas.subconjunto == "carb_60min") & (metricas.participante == "AGREGADO")]["n_exemplos"].iloc[0]
print("Tamanho do subconjunto com carboidrato em (t-60, t] no teste:", n_carb, "(VAZIO!)" if n_carb == 0 else "")

## 8. Gráfico: primeira janela de teste com ≥ 2 h

Pegamos o primeiro participante elegível que tenha 24 origens consecutivas no teste e alinhamos as previsões ao **horário do alvo** (origem + 30 min), que é o instante em que elas valem.

In [ ]:
fig_done = False
for sid in eligible:
    d = ex[(ex.subject == sid) & (ex.particao == "teste")].sort_values("origin_idx")
    run = first_run(d["origin_idx"].values, 24)
    if run is None: continue
    w = d.iloc[run[:24]]
    plt.figure(figsize=(11, 4))
    plt.plot(w["t_target"], w["y"], "k-o", ms=3, label="glicemia real (alvo)")
    plt.plot(w["t_target"], w["pred_persistencia"], "--", label="persistência")
    plt.plot(w["t_target"], w["pred_arvores_cgm"], label="árvores CGM")
    plt.plot(w["t_target"], w["pred_arvores_cgm_eventos"], label="árvores CGM + eventos")
    plt.title(f"Subject {sid} — primeira janela de teste (≥2 h); previsões alinhadas ao horário do alvo")
    plt.ylabel("mg/dL"); plt.legend(); plt.xticks(rotation=30); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "comparacao.png"), dpi=130); plt.show()
    fig_done = True; break
if not fig_done: print("Nenhum participante com janela de teste contínua de 2 h.")

## 9. Salvar tudo

Previsões, modelos treinados, cortes temporais, configuração com versões das bibliotecas e um `resumo.md` com a conclusão calculada a partir dos números reais.

In [ ]:
cols = ["subject", "t", "t_target", "g_0", "y", "particao"] + list(MODELS.values())
ex[cols].rename(columns={"subject": "participante", "t": "instante_previsao", "t_target": "instante_alvo",
                         "g_0": "glicemia_atual", "y": "alvo_real"}).to_csv(os.path.join(OUT_DIR, "previsoes.csv"), index=False)

joblib.dump(m_cgm, os.path.join(OUT_DIR, "modelo_arvores_cgm.joblib"))
joblib.dump(m_ev, os.path.join(OUT_DIR, "modelo_arvores_cgm_eventos.joblib"))
json.dump(cut_times, open(os.path.join(OUT_DIR, "cortes_temporais.json"), "w"), indent=1, ensure_ascii=False)

versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "matplotlib": __import__("matplotlib").__version__}
config = dict(QUICK=QUICK, SEED=SEED, participantes_usados=eligible, descartados=discarded,
              hgb=dict(max_iter=100, max_leaf_nodes=15, learning_rate=0.05, early_stopping=False, random_state=42),
              treino_usado=len(train), features_cgm=FEAT_CGM, features_eventos=FEAT_EV, versoes=versions)
json.dump(config, open(os.path.join(OUT_DIR, "config.json"), "w"), indent=1, ensure_ascii=False)

a = metricas[(metricas.particao == "teste") & (metricas.participante == "AGREGADO") & (metricas.subconjunto == "todos")].set_index("modelo")
gain = a.loc["arvores_cgm", "MAE"] - a.loc["arvores_cgm_eventos", "MAE"]
rel = 100 * gain / a.loc["arvores_cgm", "MAE"]
if gain > 0:
    concl = f"Os eventos reduziram o MAE de teste em {gain:.2f} mg/dL ({rel:.1f}%) em relação às árvores só com CGM."
else:
    concl = f"Os eventos NÃO melhoraram o MAE de teste (diferença de {gain:.2f} mg/dL); resultado mantido e reportado."
if a.loc["arvores_cgm_eventos", "MAE"] > a.loc["persistencia", "MAE"]:
    concl += " As árvores perderam para a persistência no MAE de teste."
print(concl)

resumo = f"""# Resumo — Protótipo 1 (árvores, previsão em 30 min)

- Modo: {'rápido' if QUICK else 'completo'} | participantes: {eligible} | descartados: {discarded}
- Exemplos por partição: {ex.groupby('particao').size().to_dict()} (treino usado: {len(train)})
- Tempos: preparação {t_prep_load + t_prep_ex:.1f}s, treino {t_train:.1f}s, inferência {t_infer:.1f}s
- Subconjunto com carboidrato em (t-60, t] no teste: {n_carb} exemplos

## Tabela principal (teste, agregado, todos os instantes)

{md_table(a.reset_index()[['modelo','n_exemplos','MAE','RMSE']])}

## Conclusão

{concl}

Limitações: desenho retrospectivo para pessoas já observadas; zero em carboidrato/bolus = nenhum registro;
somas são resumos de registros, não carboidrato/insulina ativos.
"""
open(os.path.join(OUT_DIR, "resumo.md"), "w").write(resumo)
print(os.listdir(OUT_DIR))